# BM25 Keyword Search Fundamentals [Step 1 - Classic Information Retrieval]

> **MLCourse - Agentic AI - Hybrid Search**

This notebook explains BM25, the gold standard for keyword-based search.
We will cover tokenization, TF-IDF scoring, and the BM25 algorithm
itself. We use the `rank_bm25` library to score passages from Alice
in Wonderland and show when BM25 excels versus when it struggles.

### Import all libraries needed for this notebook.


In [ ]:
import re                              # Regular expressions for tokenization
import math                            # Logarithms for TF-IDF math
from collections import Counter        # Term frequency counting


### Part 1: Loading the Corpus


In [ ]:
# We load Alice in Wonderland and split it into paragraph-sized chunks.

CORPUS_PATH = r"D:\projects\python\MLCourse\03_agentic_ai\data\alice.txt"

with open(CORPUS_PATH, "r", encoding="utf-8") as f:
    raw_text = f.read()

# Split on double newlines to get paragraphs, filter short ones.
raw_paragraphs = raw_text.split("\n\n")
corpus = [p.strip() for p in raw_paragraphs if len(p.strip()) > 80]

print(f"Loaded {len(corpus)} paragraphs from Alice in Wonderland")
print(f"First paragraph preview: {corpus[0][:100]}...")


### Part 2: Tokenization


In [ ]:
# BM25 starts with tokenization: splitting text into lowercase word tokens.
# A good tokenizer handles punctuation, case, and whitespace.

def simple_tokenize(text):
    """Lowercase and split on non-alphanumeric characters."""
    text = text.lower()
    tokens = re.findall(r"[a-z0-9]+", text)
    return tokens

# Demonstrate tokenization on a sample sentence.
sample = "The White Rabbit was late! He hurried down the hole."
tokens = simple_tokenize(sample)
print(f"Original:  {sample}")
print(f"Tokens:    {tokens}")


### Tokenize the entire corpus for later use.


In [ ]:
tokenized_corpus = [simple_tokenize(doc) for doc in corpus]
print(f"Tokenized {len(tokenized_corpus)} documents")
print(f"Example doc has {len(tokenized_corpus[0])} tokens")


### Part 3: Term Frequency (TF)


In [ ]:
# TF measures how often a term appears in a document.
# Raw count: TF(t, d) = count of t in d
# Normalized: TF(t, d) = count(t, d) / len(d)
#
# Higher TF means the term is more important to that document.

doc_tokens = tokenized_corpus[0]
term_counts = Counter(doc_tokens)

print("Term frequencies in first paragraph:")
for term, count in term_counts.most_common(10):
    tf_raw = count
    tf_norm = count / len(doc_tokens)
    print(f"  '{term}': raw={tf_raw}, normalized={tf_norm:.4f}")


### Part 4: Inverse Document Frequency ( IDF )


In [ ]:
# IDF measures how rare a term is across the entire corpus.
# IDF(t) = log( N / df(t) ) where:
#   N = total number of documents
#   df(t) = number of documents containing term t
#
# Rare terms get higher IDF -- they are more informative.

N = len(tokenized_corpus)

# Build document frequency map.
doc_freq = {}
for doc in tokenized_corpus:
    unique_terms = set(doc)
    for term in unique_terms:
        doc_freq[term] = doc_freq.get(term, 0) + 1

# Show IDF for some interesting terms.
test_terms = ["the", "rabbit", "alice", "croquet", "cheshire"]
print("Inverse Document Frequency values:")
for term in test_terms:
    df = doc_freq.get(term, 1)
    idf = math.log(N / df)
    print(f"  '{term}': df={df}, idf={idf:.4f}")


In [7]:
# "the" appears in almost every document so it has low IDF.
# "cheshire" appears in few documents so it has high IDF.
# This is exactly the intuition: rare words are more informative.

### Part 5: TF-IDF Scoring


In [ ]:
# TF-IDF combines TF and IDF into a single score.
# TF-IDF(t, d) = TF(t, d) * IDF(t)
#
# A term gets a high score when it appears frequently in a document
# but rarely across the corpus.

print("TF-IDF scores for selected terms in first paragraph:")
doc_len = len(doc_tokens)
for term in ["rabbit", "alice", "the", "hole"]:
    tf = term_counts.get(term, 0) / doc_len
    df = doc_freq.get(term, 1)
    idf = math.log(N / df)
    tfidf = tf * idf
    print(f"  '{term}': tf={tf:.4f}, idf={idf:.4f}, tfidf={tfidf:.6f}")


### Part 6: The BM25 Algorithm


In [ ]:
# BM25 improves on TF-IDF with two key ideas:
#
# 1. Saturation: increasing TF has diminishing returns.
#    A term appearing 10 times is not 10x more relevant than 5 times.
#    BM25 uses a saturation function with parameter k1.
#
# 2. Length normalization: longer documents naturally have higher TF.
#    BM25 normalizes by document length relative to average length.
#    Parameter b controls the strength of this normalization.
#
# BM25 formula for a query Q and document d:
#   score(d, Q) = SUM over t in Q of [ IDF(t) * (tf(t,d) * (k1 + 1)) / (tf(t,d) + k1 * (1 - b + b * |d|/avgdl)) ]
#
# Default parameters: k1=1.5, b=0.75

from rank_bm25 import BM25Okapi          # Production BM25 implementation

# Build the BM25 index over our tokenized corpus.
bm25 = BM25Okapi(tokenized_corpus)

print(f"BM25 index built with {len(tokenized_corpus)} documents")
print(f"Parameters: k1=1.5, b=0.75 (defaults)")


### Part 7: Scoring Documents with BM25


In [ ]:
# We query the index with different search terms and see which
# documents score highest.

query = "white rabbit"
query_tokens = simple_tokenize(query)
print(f"Query: '{query}'")
print(f"Tokens: {query_tokens}")
print()

scores = bm25.get_scores(query_tokens)
top_indices = scores.argsort()[::-1][:5]

print("Top 5 results:")
for rank, idx in enumerate(top_indices, 1):
    score = scores[idx]
    preview = corpus[idx][:80].replace("\n", " ")
    print(f"  #{rank} (score={score:.4f}): {preview}...")


### Part 8: BM25 with a Multi-Term Query


In [ ]:
# BM25 handles multi-term queries by summing scores across all query terms.

query2 = "queen croquet ground"
query2_tokens = simple_tokenize(query2)
print(f"Query: '{query2}'")
print(f"Tokens: {query2_tokens}")
print()

scores2 = bm25.get_scores(query2_tokens)
top_indices2 = scores2.argsort()[::-1][:5]

print("Top 5 results:")
for rank, idx in enumerate(top_indices2, 1):
    score = scores2[idx]
    preview = corpus[idx][:80].replace("\n", " ")
    print(f"  #{rank} (score={score:.4f}): {preview}...")


### Part 9: When BM25 Works Best


In [ ]:
# BM25 excels in these scenarios:
#
# 1. **Exact keyword matches** -- searching for "croquet" finds documents
#    with that exact word, regardless of context.
# 2. **Rare/specific terms** -- technical jargon, proper nouns, codes.
# 3. **Speed** -- BM25 is extremely fast, no neural network needed.
# 4. **No training data** -- works out of the box with zero supervision.
# 5. **Short queries** -- users typing a few keywords.
#
# BM25 struggles with:
# 1. **Synonyms** -- "happy" will not match "joyful" or "content".
# 2. **Semantic meaning** -- "animals that hop" misses "rabbit".
# 3. **Long natural language queries** -- verbose questions confuse it.
# 4. **Cross-lingual search** -- only works in one language.

print("When BM25 works best:")
print("  + Exact keyword matches")
print("  + Rare or domain-specific terms")
print("  + Fast retrieval with no GPU needed")
print("  + Zero-shot: no training required")
print()
print("When BM25 struggles:")
print("  - Synonym matching (happy vs joyful)")
print("  - Semantic understanding (animals that hop vs rabbit)")
print("  - Verbose natural language queries")
print("  - Cross-lingual search")


### Part 10: BM25 Parameter Sensitivity


In [ ]:
# Let us see how k1 and b affect results.

from rank_bm25 import BM25Okapi

# Try different k1 values (saturation control).
print("Effect of k1 on scoring (query: 'rabbit'):")
for k1_val in [0.5, 1.5, 3.0]:
    bm25_custom = BM25Okapi(tokenized_corpus, k1=k1_val, b=0.75)
    scores_k = bm25_custom.get_scores(query_tokens)
    top_idx = scores_k.argmax()
    print(f"  k1={k1_val}: top doc score={scores_k[top_idx]:.4f}")

print()
print("Effect of b on scoring (query: 'rabbit'):")
for b_val in [0.0, 0.5, 0.75, 1.0]:
    bm25_custom = BM25Okapi(tokenized_corpus, k1=1.5, b=b_val)
    scores_b = bm25_custom.get_scores(query_tokens)
    top_idx = scores_b.argmax()
    print(f"  b={b_val}: top doc score={scores_b[top_idx]:.4f}")


In [14]:
# b=0 means no length normalization; b=1 means full normalization.
# k1 controls how quickly TF saturates: low k1 saturates fast (less weight
# on repeated terms), high k1 behaves more like raw TF.
#
# In the next notebook we explore dense vector retrieval, which captures
# semantic meaning that BM25 misses.

print("Key takeaways:")
print("  BM25 is a term-frequency based scoring algorithm")
print("  It improves on TF-IDF with saturation and length normalization")
print("  It is fast, unsupervised, and excellent for keyword search")
print("  It does not understand synonyms or semantic similarity")
print("  Combining BM25 with dense retrieval gives the best of both worlds")

Key takeaways:
  BM25 is a term-frequency based scoring algorithm
  It improves on TF-IDF with saturation and length normalization
  It is fast, unsupervised, and excellent for keyword search
  It does not understand synonyms or semantic similarity
  Combining BM25 with dense retrieval gives the best of both worlds
